# Chapter 1 · Text is numbers

A language model never sees letters. It only sees numbers. In this chapter you turn
Shakespeare into numbers and back again, and you meet the number this whole project is
about: **4.17**, the score of a model that knows nothing.

| | |
|---|---|
| **Open box** | the tokenizer, `tinygpt/tokenizer.py` |
| **The number** | 4.17 → 3.35 |
| **Time** | about an hour |
| **Python you need** | strings, lists, dictionaries, `for` loops |
| **GPU** | not needed for this chapter |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    root = Path("/content/tinyGPT")
    if not root.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(root))
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

You will work with **tinyshakespeare**: about 40,000 lines from Shakespeare's plays, in a
single text file. Before you look at it, write down three guesses.

Think about lower-case letters, capital letters, digits, punctuation, spaces and line breaks.

In [ ]:
# Type your guesses, then run this cell. Any guess is fine.
guess_n_distinct = None     # how many *different* characters does the file use? e.g. 80
guess_most_common = None    # which single character appears most often? e.g. "e"
guess_rarest = None         # which character appears least often?

## 2 · Run `everyone`

Load the text and look at the start of it.

In [ ]:
text = tinygpt.load_text()
print(f"{len(text):,} characters")
print(text[:300])

Now build a **tokenizer** from the text. It finds every distinct character and gives each
one a number.

In [ ]:
tok = tinygpt.CharTokenizer.from_text(text)
print(tok.vocab_size, "different characters:")
print(tok.chars)
print("your guess:", guess_n_distinct)

`tok.chars` is the model's whole alphabet, called its **vocabulary**. The first two
entries look odd: `'\n'` is the line break and `' '` is the space.

Which characters are common, and which are rare?

In [ ]:
tinygpt.char_counts(text)
print("your guesses:", guess_most_common, guess_rarest)

The tokenizer's job: text → numbers → text.

In [ ]:
ids = tok.encode("To be, or not to be")
print(ids)
print(tok.decode(ids))

## 3 · Investigate `everyone`

### Open the box

Everything above came from one small class. This is its code, exactly as it ran:

In [ ]:
tinygpt.show(tinygpt.CharTokenizer)

Answer these in the text cell below, or just in your head.

1. `from_text` uses `sorted(set(text))`. What does `set` do here? Why sort?
2. Why is `'\n'` number 0 and `' '` number 1? (The next cell has a hint: `ord` gives
   each character's code in the Unicode table, and `sorted` puts characters in that order.)
3. `self.stoi` turns a character into its number. What turns a number back into a character?
4. What does `encode` do with a character that is not in the vocabulary? Find the line.

In [ ]:
print(tok.stoi["T"], tok.stoi["o"])        # character -> number
print(tok.chars[32], tok.chars[53])        # number -> character
print(ord("\n"), ord(" "), ord("A"), ord("a"))

*Your explanation (double-click to edit):*

### The number

A language model's job is to look at the text so far and give a probability to every
character that could come next. Its **loss** is how surprised it is, on average, by the
character that really came next:

> surprise = −log(the probability the model gave to the right character)

Certain and right: −log(1) = 0, no surprise at all. A model that gave the right
character a 1 in 65 chance: −log(1/65).

**Predict:** a model that knows nothing gives all 65 characters the same chance, 1/65.
What is its loss? Then run the cell.

In [ ]:
import math
print(-math.log(1 / 65))

**4.17.** Every model in this project starts here, and our job is to push this number
down. (The unit is the *nat*, because we use the natural logarithm.)

Can we beat 4.17 with no cleverness at all? Here is a model that only knows **how common
each character is**: 8.5% of the text is the letter e, so it always bets 8.5% on e.

We count on the first 90% of the text and measure on the last 10%, which the counting
never saw. That last part is the **validation** text; measuring on unseen text is the
only fair test.

**Predict:** will this land above or below 4.17? By how much?

In [ ]:
from collections import Counter

n = int(len(text) * 0.9)
train_text, val_text = text[:n], text[n:]     # learn from the first 90%, test on the last 10%

freq = Counter(train_text)
surprise = [-math.log(freq[ch] / len(train_text)) for ch in val_text]
freq_loss = sum(surprise) / len(surprise)
print(f"knowing frequencies: {freq_loss:.2f}")

Knowing frequencies moves the number from 4.17 to **3.35**. It still knows nothing about
*order*: it would bet on e right after another e. Chapter 2 adds order.

## 4 · Modify `most people`

**A.** Encode your own name. Then try a word with an accent, like `"café"`, or an emoji.
**Predict** what will happen before you run the second cell.

In [ ]:
print(tok.encode("Hermione"))     # change this to your name

In [ ]:
try:
    print(tok.encode("café"))     # predict first!
except ValueError as error:       # catch the error so the notebook keeps going
    print("ValueError:", error)

Why can this model never *write* the letter é, however well it is trained?

*Your explanation (double-click to edit):*

**B.** Make everything lower-case. **Predict:** how big will the vocabulary be? What is
the knowing-nothing number for that vocabulary?

In [ ]:
low = text.lower()
low_tok = tinygpt.CharTokenizer.from_text(low)
print(low_tok.vocab_size, "different characters")
print("knowing nothing:", -math.log(1 / low_tok.vocab_size))

The knowing-nothing number dropped from 4.17 to 3.66 without any learning. Is a
lower-case model *better*? What did it give up? What does this tell you about comparing
loss numbers between models with different vocabularies?

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Build your own tokenizer without the class: two dictionaries and two functions. Fill in
the blanks, then run the cell. The check compares your functions with tinyGPT's on a few
pieces of text and tells you what, if anything, is wrong.

In [ ]:
chars = sorted(set(text))      # the vocabulary: every distinct character, in order

stoi = {}                      # character -> number: fill this in (a for loop over enumerate(chars) works)
itos = {}                      # number -> character: fill this in too

def my_encode(s):
    """Text -> list of numbers."""
    return []                  # replace this line

def my_decode(ids):
    """List of numbers -> text."""
    return ""                  # replace this line

tinygpt.check.tokenizer(my_encode, my_decode)

When the check passes, you have written the first piece of a language model.

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/01_text_is_numbers.md) (also in the `solutions/` folder of your copy).

## Where the number stands

Your result on the line from knowing nothing (4.17) to the finished tinyGPT (1.49):

In [ ]:
tinygpt.scoreboard({"knowing frequencies": freq_loss})

## Common questions

**Why characters and not words?** A character vocabulary is tiny (65) and can spell any
word in the text. A word vocabulary would need tens of thousands of entries and still
could not handle a word it had never seen.

**Do ChatGPT and Claude use characters?** No. They use *tokens*: chunks of text, often
pieces of words, from a vocabulary of roughly 50,000 to 200,000. The idea is the same as
here: text becomes numbers before the model sees it, and numbers become text afterwards.

**Why a logarithm?** It turns "how unlikely" into "how surprised" with the right
properties: a sure thing costs 0, an impossible thing costs infinity, and the surprise of
two events in a row is the sum of their surprises.

**What is a nat?** The unit of surprise when you use the natural logarithm. With log
base 2 the unit is the *bit*: 4.17 nats is 6.02 bits, the number of yes/no questions
needed to pick one of 65 characters.

## If something breaks

| What you see | What to do |
|---|---|
| `NameError: name 'tinygpt' is not defined` | Run the setup cell. You need to after every restart. |
| `NameError` for `text`, `tok` or another name | Run the cells above it, in order. In Colab: Runtime > Run before. |
| The setup cell fails while cloning | Check your internet connection, then run it again. |
| `ValueError: ... is not in the vocabulary` | Expected in step 4: that character is not one of the 65. |

## Next

**Chapter 2 · Counting beats guessing.** Use the previous character to predict the next
one, by counting pairs. The number drops to 2.48.